# Adaptive Recursive RAG with Binary Evidence Verification

This notebook implements and benchmarks three RAG architectures:
1. **Standard RAG**: Retrieves once and generates.
2. **Always Recursive RAG**: Forces exactly 3 retrieval loops.
3. **Adaptive Recursive RAG**: Retrieves, extracts claims, and verifies against context. Dynamically refines the query and loops up to 2 times only if necessary. Can abstain safely.

---

## 1. Setup & Imports
Installing and importing necessary libraries (`faiss-cpu`, `bitsandbytes`, `transformers`, `datasets`, `sentence-transformers`, `matplotlib`, `pandas`).

In [ ]:
!pip install -q \
transformers==4.41.2 \
accelerate==0.30.1 \
bitsandbytes==0.43.1 \
sentence-transformers==2.7.0 \
faiss-cpu \
datasets \
pandas \
matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 58.1 MB/s eta 0:00:00


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

import faiss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import torch

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


## 2. Dataset Loading & Corpus Construction

In [ ]:
print("Loading HotpotQA (distractor) validation split, sample 75...")
dataset = load_dataset("hotpot_qa", "distractor", split="validation")
sample = dataset.select(range(75))

print("Constructing corpus with deduplication...")
all_passages = []
seen = set()

for example in sample:
    titles = example["context"]["title"]
    sentences_list = example["context"]["sentences"]
    for title, sentences in zip(titles, sentences_list):
        passage = title + ". " + " ".join(sentences)
        if passage not in seen:
            seen.add(passage)
            all_passages.append(passage)

print(f"Sample size: {len(sample)} questions.")
print(f"Corpus size: {len(all_passages)} unique passages built for retrieval.")

Loading HotpotQA (distractor) validation split, sample 75...


README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Constructing corpus with deduplication...
Sample size: 75 questions.
Corpus size: 741 unique passages built for retrieval.


## 3. Embedding Model & Vector Index (FAISS)

In [ ]:
print("Loading BGE-small-en-v1.5 embedding model...")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

print("Embedding passages...")
passage_embeddings = embed_model.encode(
    all_passages,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True
)

print("Building FAISS IndexFlatIP...")
dimension = passage_embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dimension)
index.add(passage_embeddings)
print(f"Index built with {index.ntotal} vectors.")

def embed_query(query):
    prefixed_query = "Represent this sentence for searching relevant passages: " + query
    return embed_model.encode([prefixed_query], normalize_embeddings=True)

def retrieve(query, existing_docs, k=3):
    query_emb = embed_query(query)
    # query_emb is already normalized
    scores, indices = index.search(query_emb, k)
    new_docs = [all_passages[i] for i in indices[0]]
    for doc in new_docs:
        if doc not in existing_docs:
            existing_docs.append(doc)
    return existing_docs

Loading BGE-small-en-v1.5 embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding passages...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Building FAISS IndexFlatIP...
Index built with 741 vectors.


## 4. Language Model & Core Functions (generation, extraction, verification)

In [ ]:
print("Loading Phi-3 Mini in 4-bit...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_id = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

llm = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager"
)

llm.eval()

def llm_call(system_message, user_message, max_new_tokens):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(llm.device) for k, v in inputs.items()}
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            do_sample=False,
            repetition_penalty=1.1
        )

    generated_tokens = outputs[0][input_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# ── LLM Call Tracker ─────────────────────────────────────────────────────
_llm_call_count = {"value": 0}

def reset_llm_counter():
    _llm_call_count["value"] = 0

def get_llm_count():
    return _llm_call_count["value"]

# Monkey-patch llm_call to auto-count
_original_llm_call = llm_call

def llm_call(system_message, user_message, max_new_tokens):
    _llm_call_count["value"] += 1
    return _original_llm_call(system_message, user_message, max_new_tokens)



def generate(query, docs):
    system_msg = (
        "You are a factual question answering assistant.\n"
        "You will be given a question and a set of reference documents.\n"
        "Answer the question using only the information in the documents.\n"
        "Be concise. Do not add information that is not in the documents.\n"
        "If the documents do not contain enough information, say:\n"
        "\"I cannot determine this from the provided documents.\""
    )

    docs_text = "\n\n".join(docs)
    user_msg = f"Documents:\n{docs_text}\n\nQuestion: {query}\n\nAnswer:"
    return llm_call(system_msg, user_msg, max_new_tokens=256)

ABSTAIN_PHRASES = [
    "cannot determine", "cannot find", "not enough information",
    "insufficient information", "not provided in the documents",
    "not mentioned in the documents", "i don\t know",
    "unable to determine", "no information", "not stated in"
]

def is_abstain_answer(answer):
    ans_lower = answer.lower().strip()
    return any(phrase in ans_lower for phrase in ABSTAIN_PHRASES)

def extract_claims(answer):
    system_msg = (
        "You are a precise factual analyzer.\n"
        "Your job is to break a given answer into individual factual claims.\n"
        "Each claim must contain exactly ONE fact — do not combine multiple facts into one claim.\n"
        "Each claim must be a single standalone statement that can be\n"
        "verified independently.\n"
        "Output exactly 2 to 3 claims, one per line, numbered.\n"
        "Do not output anything else — no explanation, no preamble,\n"
        "no extra text."
    )

    user_msg = f"Answer: {answer}\n\nBreak this answer into 2 to 3 factual claims, one per line, numbered:\n1."
    raw_output = llm_call(system_msg, user_msg, max_new_tokens=128)
    raw_output = "1. " + raw_output

    import re
    claims = []
    lines = raw_output.split("\n")
    for line in lines:
        cleaned = line.strip()
        if not cleaned: continue
        cleaned = re.sub(r"^\d+[\.\)\-]\s*", "", cleaned)
        cleaned = re.sub(r"^[\-\*]\s*", "", cleaned)
        cleaned = re.sub(r"\s*\(Claim\s*#?\d+\)\s*$", "", cleaned, flags=re.IGNORECASE).strip()

        if cleaned and len(cleaned) > 15:
            claims.append(cleaned)

    if len(claims) > 3:
        claims = claims[:3]

    if len(claims) < 2:
        sentences = re.split(r"[\.\?\!]", answer)
        claims = [s.strip() for s in sentences if len(s.strip()) > 15][:3]

    return claims


def verify(claims, docs):
    system_msg = (
        "You are a strict fact verification assistant.\n"
        "You will be given a factual claim and a set of reference documents.\n"
        "Your job is to determine if the claim is supported by the documents.\n"
        "Reply with exactly one word: Yes or No.\n"
        "Do not explain. Do not add any other text. Just Yes or No."
    )

    docs_text = "\n\n".join(docs)
    results = []
    for claim in claims:
        user_msg = f"Documents:\n{docs_text}\n\nClaim: {claim}\n\nIs this claim supported by the documents above? Answer Yes or No:"
        raw_output = llm_call(system_msg, user_msg, max_new_tokens=8).lower().strip()
        res = False
        if raw_output.startswith("yes"):
            res = True
        elif raw_output.startswith("no"):
            res = False
        elif "yes" in raw_output[:20]:
            res = True
        elif "no" in raw_output[:20]:
            res = False
        results.append(res)
    return results

def refine_query(original_query, failed_claim):
    system_msg = "You are an intelligent search assistant."
    user_msg = f"Generate a concise search query to retrieve factual information.\nKeep key entities from the original question.\n\nOriginal question: {original_query}\nClaim: {failed_claim}\n\nSearch query:"
    raw = llm_call(system_msg, user_msg, max_new_tokens=32)
    return raw.strip()

def filter_claims(claims, answer):
    """
    Filter claims to those that are substantive and grounded.
    Removes claims that are too short, are generic filler, or
    do not contain any named entity or number from the answer.
    """
    import re
    # Extract tokens of length >= 4 from answer (likely meaningful words)
    meaningful_tokens = set(
        t.lower() for t in re.findall(r'\b\w{4,}\b', answer)
    )
    filtered = []
    for claim in claims:
        claim_lower = claim.lower()
        # Must be substantive length
        if len(claim.split()) < 5:
            continue
        # Must share at least one meaningful token with the answer
        if not any(tok in claim_lower for tok in meaningful_tokens):
            continue
        filtered.append(claim)
    # Fallback: if filter removes everything, return original claims
    return filtered if filtered else claims

def normalize(text):
    return "".join(c.lower() for c in text if c.isalnum() or c.isspace()).strip()


def get_verification_threshold(n_claims):
    """
    Strict tiered threshold — requires strong evidence support.
    n=1 → all must pass (1/1)
    n=2 → all must pass (2/2)
    n=3 → at most 1 failure allowed (2/3)
    """
    if n_claims <= 2:
        return n_claims        # all must pass
    else:
        return n_claims - 1    # at most 1 failure


Loading Phi-3 Mini in 4-bit...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

## 5. Pipeline Orchestrator (The Three Modes)

In [ ]:
def f1_score(answer, ground_truth):
    """
    Token-level F1 overlap between predicted answer and ground truth.
    Standard metric for HotpotQA — handles paraphrased correct answers
    that exact match would score as 0.
    """
    if not isinstance(answer, str) or not isinstance(ground_truth, str):
        return 0.0
    pred_tokens = normalize(answer).split()
    gt_tokens   = normalize(ground_truth).split()
    if not pred_tokens or not gt_tokens:
        return 0.0
    common = set(pred_tokens) & set(gt_tokens)
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall    = len(common) / len(gt_tokens)
    return round(2 * precision * recall / (precision + recall), 4)

def check_correctness(answer, ground_truth):
    if not isinstance(answer, str) or not isinstance(ground_truth, str):
        return False
    ans_clean = normalize(answer)
    gt_clean = normalize(ground_truth)
    return gt_clean in ans_clean

def run_pipeline(query, ground_truth, mode="standard"):
    start_time = time.time()
    reset_llm_counter()
    result = {
        "answer": "",
        "steps": 0,
        "verified": False,
        "abstained": False,
        "mode": mode,
        "correct": False,
        "f1": 0.0,
        "latency": 0.0,
        "llm_calls": 0,
        "verified_claims": 0,
        "total_claims": 0
    }

    docs = []

    if mode == "standard":
        docs = retrieve(query, docs, k=8)
        answer = generate(query, docs)
        result["answer"] = answer
        result["steps"] = 1

    elif mode == "recursive":
        k_schedule = [8, 9, 10]
        for loop_idx in range(3):
            docs = retrieve(query, docs, k=k_schedule[loop_idx])
            answer = generate(query, docs)
        result["answer"] = answer
        result["steps"] = 3

    elif mode == "adaptive":
        current_query = query
        MAX_DEPTH = 2
        for step in range(1, MAX_DEPTH + 1):
            k_val = 8 + (step - 1)
            docs = retrieve(current_query, docs, k=k_val)
            answer = generate(query, docs) # Generate uses original query
            result["steps"] = step

            raw_claims = extract_claims(answer)
            claims = filter_claims(raw_claims, answer)
            if not claims:
                claims = raw_claims

            verifications = verify(claims, docs)
            threshold = get_verification_threshold(len(verifications))
            verified_sum = sum(verifications)

            if is_abstain_answer(answer) or verified_sum < threshold:
                if step == MAX_DEPTH:
                    result["abstained"]       = True
                    result["verified_claims"] = int(sum(verifications))
                    result["total_claims"]    = int(len(verifications))
                    result["answer"]          = "Cannot find sufficient information." if is_abstain_answer(answer) else ""
                    break
                else:
                    if is_abstain_answer(answer):
                        current_query = query
                    else:
                        failed_claims = [c for c, v in zip(claims, verifications) if not v]
                        if failed_claims:
                            current_query = refine_query(query, failed_claims[0])
                        else:
                            current_query = query
                    continue
            else:
                result["verified"]        = True
                result["verified_claims"] = int(sum(verifications))
                result["total_claims"]    = int(len(verifications))
                result["answer"]          = answer
                break

    result["llm_calls"] = get_llm_count()
    result["latency"] = round(time.time() - start_time, 2)
    if not result["abstained"]:
        result["correct"] = check_correctness(result["answer"], ground_truth)
        result["f1"]      = f1_score(result["answer"], ground_truth)
    else:
        result["correct"] = False
        result["f1"]      = 0.0

    # Free GPU memory after each pipeline run
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


## 6. Demo Execution

In [ ]:
# Demo Cell - Trace execution manually for verbosity

def match_query_to_dataset(target_query, sample):
    queries = [ex["question"] for ex in sample]
    q_emb = embed_model.encode([target_query], normalize_embeddings=True)
    db_emb = embed_model.encode(queries, normalize_embeddings=True)
    scores = np.dot(db_emb, q_emb.T).squeeze()
    idx = np.argmax(scores)
    return sample[int(idx)]

target_query = "What WB supernatrual drama series did Rose Mcgowan star in?"
match = match_query_to_dataset(target_query, sample)

demo_q = match["question"]
demo_gt = match["answer"]

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"QUERY: {demo_q}")
print(f"GROUND TRUTH: {demo_gt}")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n")

# STANDARD MODE
print("MODE: standard")
docs = retrieve(demo_q, [], k=8)
ans = generate(demo_q, docs)
print(f"  Retrieved docs: {len(docs)}")
print(f"  Answer: \"{ans}\"")
print(f"  Steps: 1")
print(f"  Verified: False (not checked)")
print(f"  Abstained: False\n")

# RECURSIVE MODE
print("MODE: recursive")
docs = []
k_schedule = [8, 9, 10]
for idx in range(3):
    docs = retrieve(demo_q, docs, k=k_schedule[idx])
    ans = generate(demo_q, docs)
print(f"  Retrieved docs: {len(docs)}")
print(f"  Answer: \"{ans}\"")
print(f"  Steps: 3")
print(f"  Verified: False (not checked)")
print(f"  Abstained: False\n")

# ADAPTIVE MODE
print("MODE: adaptive")
docs = []
current_query = demo_q
MAX_DEPTH = 2
final_ans = ""
final_step = 0
abstained = False

for step in range(1, MAX_DEPTH + 1):
    k_val = 8 + (step - 1)
    docs = retrieve(current_query, docs, k=k_val)
    ans = generate(demo_q, docs)
    final_ans = ans
    final_step = step

    print(f"  --- Step {step} ---")
    if current_query != demo_q:
        print(f"  [Refined Search Query]: {current_query}")

    print(f"  Retrieved docs so far: {len(docs)}")
    print(f"  Generated Answer: \"{ans}\"")

    if is_abstain_answer(ans):
        print("  -> Model abstained directly.")
        if step == MAX_DEPTH:
            abstained = True
            final_ans = "Cannot find sufficient information."
            break
        else:
            current_query = demo_q
            print("  -> Getting more docs using original query...")
            continue

    raw_claims = extract_claims(ans)
    claims = filter_claims(raw_claims, ans)
    if not claims:
        claims = raw_claims

    verifs = verify(claims, docs)
    print(f"  Extracted & Filtered Claims:")
    for c, v in zip(claims, verifs):
        print(f"    - {c} -> Supported: {v}")

    threshold = get_verification_threshold(len(verifs))
    verified_sum = sum(verifs)

    if verified_sum >= threshold:
        print(f"  -> {verified_sum}/{len(verifs)} claims supported. Majority threshold met! Stopping early.")
        break
    else:
        if step == MAX_DEPTH:
            print(f"  -> Majority not met ({verified_sum}/{len(verifs)}). Max depth reached. Abstaining.")
            abstained = True
        else:
            failed = [c for c, v in zip(claims, verifs) if not v]
            if failed:
                print(f"  -> Majority not met. Refining query based on failed claim: \"{failed[0]}\"")
                current_query = refine_query(demo_q, failed[0])

print("\n  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  ADAPTIVE FINAL SUMMARY")
print(f"  Final Answer : {final_ans}")
print(f"  Total Steps  : {final_step}")
print(f"  Abstained    : {abstained}")
print(f"  Verified     : {not abstained}")
ans_clean = normalize(final_ans)
gt_clean = normalize(demo_gt)
correct = gt_clean in ans_clean
print(f"  Correct      : {correct}")
print(f"  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
QUERY: What WB supernatrual drama series was Jawbreaker star Rose Mcgowan best known for being in?
GROUND TRUTH: Charmed
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

MODE: standard


  Retrieved docs: 8
  Answer: "Rose McGowan is best known for starring in the WB supernatural drama series "Charmed"."
  Steps: 1
  Verified: False (not checked)
  Abstained: False

MODE: recursive
  Retrieved docs: 10
  Answer: "Rose McGowan is best known for starring in the WB supernatural drama series "Charmed," where she plays the character Paige Matthews."
  Steps: 3
  Verified: False (not checked)
  Abstained: False

MODE: adaptive
  --- Step 1 ---
  Retrieved docs so far: 8
  Generated Answer: "Rose McGowan is best known for starring in the WB supernatural drama series "Charmed"."
  Extracted & Filtered Claims:
    - 1. Rose McGowan starred in the WB supernatural drama series called "Charmed." -> Supported: False
    - She gained recognition from her role as part of the cast on "Charmed." -> Supported: False
  -> Majority not met. Refining query based on failed claim: "1. Rose McGowan starred in the WB supernatural drama series called "Charmed.""
  --- Step 2 ---
  [Refined Sear

## 7. Full Benchmark Run

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 7 — Full Benchmark Run (Resumable + OOM-safe)
# Runs all 75 HotpotQA questions through Standard, Recursive,
# and Adaptive RAG. Collects per-query results for analysis.
# If interrupted, re-run this cell to resume from where it stopped.
# ════════════════════════════════════════════════════════════════

import time
import json
import gc
import os

MODES   = ["standard", "recursive", "adaptive"]

# ── Resume support: load partial results if they exist ────────
if os.path.exists("benchmark_results.json"):
    with open("benchmark_results.json") as f:
        results = json.load(f)
    # Figure out how many complete questions we have
    min_done = min(len(results[m]) for m in MODES)
    print(f"Resuming from question {min_done + 1} (found {min_done} complete)")
    # Trim to only fully-complete questions
    for m in MODES:
        results[m] = results[m][:min_done]
    start_idx = min_done
else:
    results = {mode: [] for mode in MODES}
    start_idx = 0

print(f"Starting benchmark over {len(sample)} questions × {len(MODES)} modes")
print("=" * 60)

for q_idx in range(start_idx, len(sample)):
    example      = sample[q_idx]
    query        = example["question"]
    ground_truth = example["answer"]
    difficulty   = example.get("level", "unknown")

    print(f"\n[{q_idx+1:02d}/{len(sample)}] {query[:80]}...")

    for mode in MODES:
        t0  = time.time()
        res = run_pipeline(query, ground_truth, mode=mode)
        res["question"]     = query
        res["ground_truth"] = ground_truth
        res["difficulty"]   = difficulty
        res["q_idx"]        = q_idx
        results[mode].append(res)

        status = "\u2713" if res["correct"] else ("~" if res["abstained"] else "\u2717")
        print(f"  [{mode:10s}] {status}  EM={int(res['correct'])}  "
              f"F1={res['f1']:.2f}  steps={res['steps']}  "
              f"llm_calls={res['llm_calls']}  t={res['latency']}s")

    # ── Save after every question + clear GPU memory ─────────
    with open("benchmark_results.json", "w") as f:
        json.dump(results, f, indent=2)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("Benchmark complete. Results saved to benchmark_results.json")


## 8. Results & Visualization

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 8 — Results Aggregation & Visualisation
# Computes all metrics and produces publication-ready charts.
# ════════════════════════════════════════════════════════════════

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Load results if not already in memory ────────────────────────
try:
    results
except NameError:
    import json
    with open("benchmark_results.json") as f:
        results = json.load(f)

MODES        = ["standard", "recursive", "adaptive"]
MODE_LABELS  = {"standard": "Standard RAG", "recursive": "Always-Recursive", "adaptive": "Adaptive RAG"}
MODE_COLORS  = {"standard": "#4C72B0", "recursive": "#DD8452", "adaptive": "#55A868"}

# ── 8.1  Build flat DataFrame ─────────────────────────────────
rows = []
for mode in MODES:
    for r in results[mode]:
        rows.append({
            "mode":             mode,
            "q_idx":            r["q_idx"],
            "difficulty":       r.get("difficulty", "unknown"),
            "correct":          int(r["correct"]),
            "f1":               r["f1"],
            "abstained":        int(r["abstained"]),
            "steps":            r["steps"],
            "llm_calls":        r["llm_calls"],
            "latency":          r["latency"],
            "verified_claims":  r.get("verified_claims", 0),
            "total_claims":     r.get("total_claims", 0),
        })

df = pd.DataFrame(rows)

# ── 8.2  Aggregate summary table ───────────────────────────────
print("\n" + "═" * 80)
print("BENCHMARK RESULTS SUMMARY")
print("═" * 80)

summary = {}
for mode in MODES:
    m = df[df["mode"] == mode]
    n = len(m)
    answered   = m[m["abstained"] == 0]
    unsupported = 0
    if mode == "adaptive":
        answered_with_claims = m[(m["abstained"] == 0) & (m["total_claims"] > 0)]
        if len(answered_with_claims) > 0:
            unsupported = (
                answered_with_claims["total_claims"] - answered_with_claims["verified_claims"]
            ).gt(0).sum() / len(answered_with_claims) * 100

    summary[mode] = {
        "N":                    n,
        "Exact Match (%)":      round(m["correct"].mean() * 100, 1),
        "F1 Score":             round(m["f1"].mean(), 3),
        "Abstention Rate (%)":  round(m["abstained"].mean() * 100, 1),
        "Unsupported Ans. (%)": round(unsupported, 1) if mode == "adaptive" else "N/A",
        "Avg Steps":            round(m["steps"].mean(), 2),
        "Avg LLM Calls":        round(m["llm_calls"].mean(), 1),
        "Avg Latency (s)":      round(m["latency"].mean(), 1),
        "EM (answered only)":   round(answered["correct"].mean() * 100, 1) if len(answered) > 0 else 0,
    }

summary_df = pd.DataFrame(summary).T
summary_df.index = [MODE_LABELS[m] for m in summary_df.index]
print(summary_df.to_string())
print("═" * 80)

# ── 8.3  Difficulty breakdown ─────────────────────────────────
print("\nACCURACY BY DIFFICULTY LEVEL")
print("─" * 60)
for diff in ["easy", "medium", "hard"]:
    sub = df[df["difficulty"] == diff]
    if len(sub) == 0:
        continue
    print(f"\n  {diff.upper()} (n={len(sub)//len(MODES)}):")
    for mode in MODES:
        m_sub = sub[sub["mode"] == mode]
        em  = m_sub["correct"].mean() * 100
        f1  = m_sub["f1"].mean()
        print(f"    {MODE_LABELS[mode]:20s}  EM={em:.1f}%  F1={f1:.3f}")

# ── 8.4  Visualisations ───────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Adaptive RAG — Benchmark Results", fontsize=16, fontweight="bold", y=1.01)

x      = np.arange(len(MODES))
labels = [MODE_LABELS[m] for m in MODES]
colors = [MODE_COLORS[m] for m in MODES]

# Plot 1: Exact Match
ax = axes[0, 0]
vals = [summary[m]["Exact Match (%)"] for m in MODES]
bars = ax.bar(x, vals, color=colors, edgecolor="white", linewidth=0.8)
ax.set_title("Exact Match (%)", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
ax.set_ylim(0, 100); ax.set_ylabel("%")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f"{v:.1f}", ha="center", fontsize=9)

# Plot 2: F1 Score
ax = axes[0, 1]
vals = [summary[m]["F1 Score"] for m in MODES]
bars = ax.bar(x, vals, color=colors, edgecolor="white", linewidth=0.8)
ax.set_title("F1 Score (avg)", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
ax.set_ylim(0, 1); ax.set_ylabel("F1")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{v:.3f}", ha="center", fontsize=9)

# Plot 3: Avg LLM Calls (efficiency)
ax = axes[0, 2]
vals = [summary[m]["Avg LLM Calls"] for m in MODES]
bars = ax.bar(x, vals, color=colors, edgecolor="white", linewidth=0.8)
ax.set_title("Avg LLM Calls / Query", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
ax.set_ylabel("LLM Calls")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f"{v:.1f}", ha="center", fontsize=9)

# Plot 4: Avg Steps
ax = axes[1, 0]
vals = [summary[m]["Avg Steps"] for m in MODES]
bars = ax.bar(x, vals, color=colors, edgecolor="white", linewidth=0.8)
ax.set_title("Avg Retrieval Steps", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
ax.set_ylabel("Steps")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{v:.2f}", ha="center", fontsize=9)

# Plot 5: Abstention Rate
ax = axes[1, 1]
vals = [summary[m]["Abstention Rate (%)"] for m in MODES]
bars = ax.bar(x, vals, color=colors, edgecolor="white", linewidth=0.8)
ax.set_title("Abstention Rate (%)", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
ax.set_ylim(0, 100); ax.set_ylabel("%")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f"{v:.1f}", ha="center", fontsize=9)

# Plot 6: F1 by difficulty (adaptive only)
ax = axes[1, 2]
diffs  = ["easy", "medium", "hard"]
f1_by_diff = []
for diff in diffs:
    sub = df[(df["mode"] == "adaptive") & (df["difficulty"] == diff)]
    f1_by_diff.append(sub["f1"].mean() if len(sub) > 0 else 0)
diff_colors = ["#55A868", "#FFC107", "#E74C3C"]
bars = ax.bar(diffs, f1_by_diff, color=diff_colors, edgecolor="white", linewidth=0.8)
ax.set_title("Adaptive F1 by Difficulty", fontweight="bold")
ax.set_ylabel("F1"); ax.set_ylim(0, 1)
for bar, v in zip(bars, f1_by_diff):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{v:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("benchmark_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nFigure saved to benchmark_results.png")


## 9. Ablation Study: Adaptive w/o Query Refinement

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 9 — Ablation Study: Adaptive w/o Query Refinement
#
# Tests whether refine_query() actually contributes to accuracy,
# or whether the extra retrieval step alone explains improvement.
# Runs the adaptive pipeline but always uses the original query
# on retry — no intelligent query refinement.
# ════════════════════════════════════════════════════════════════

def run_adaptive_no_refinement(query, ground_truth):
    """
    Adaptive pipeline with query refinement disabled.
    On retry, always uses original query instead of refined one.
    Identical to adaptive mode in all other respects.
    """
    import time
    start_time = time.time()
    result = {
        "answer": "", "steps": 0, "verified": False, "abstained": False,
        "correct": False, "f1": 0.0, "llm_calls": 0,
        "verified_claims": 0, "total_claims": 0, "latency": 0.0,
        "mode": "adaptive_no_refine", "question": query,
        "ground_truth": ground_truth, "difficulty": "unknown"
    }

    reset_llm_counter()
    docs          = []
    MAX_DEPTH     = 2

    for step in range(1, MAX_DEPTH + 1):
        k_val   = 8 + (step - 1)
        docs    = retrieve(query, docs, k=k_val)          # always original query
        answer  = generate(query, docs)
        result["steps"] = step

        raw_claims   = extract_claims(answer)
        claims       = filter_claims(raw_claims, answer)
        if not claims:
            claims = raw_claims

        verifications = verify(claims, docs)
        threshold     = get_verification_threshold(len(verifications))
        verified_sum  = sum(verifications)

        if is_abstain_answer(answer) or verified_sum < threshold:
            if step == MAX_DEPTH:
                result["abstained"]       = True
                result["verified_claims"] = int(verified_sum)
                result["total_claims"]    = int(len(verifications))
                result["answer"]          = "Cannot find sufficient information."
                break
            # NO refine_query() call here — retry with same original query
            continue
        else:
            result["verified"]        = True
            result["verified_claims"] = int(verified_sum)
            result["total_claims"]    = int(len(verifications))
            result["answer"]          = answer
            break

    result["llm_calls"] = get_llm_count()
    result["latency"]   = round(time.time() - start_time, 2)
    if not result["abstained"]:
        result["correct"] = check_correctness(result["answer"], ground_truth)
        result["f1"]      = f1_score(result["answer"], ground_truth)

    return result

# ── Run ablation ──────────────────────────────────────────────────
print("Running ablation study: Adaptive without query refinement...")
ablation_results = []

for q_idx, example in enumerate(sample):
    query        = example["question"]
    ground_truth = example["answer"]
    difficulty   = example.get("level", "unknown")

    res = run_adaptive_no_refinement(query, ground_truth)
    res["difficulty"] = difficulty
    res["q_idx"]      = q_idx
    ablation_results.append(res)

    status = "✓" if res["correct"] else ("~" if res["abstained"] else "✗")
    print(f"  [{q_idx+1:02d}] {status}  EM={int(res['correct'])}  F1={res['f1']:.2f}")

# ── Compare adaptive full vs. adaptive no-refine ─────────────
full_adaptive = results["adaptive"]  # from Cell 7

full_em  = sum(r["correct"]  for r in full_adaptive) / len(full_adaptive) * 100
full_f1  = sum(r["f1"]       for r in full_adaptive) / len(full_adaptive)
full_abs = sum(r["abstained"] for r in full_adaptive) / len(full_adaptive) * 100

abl_em   = sum(r["correct"]  for r in ablation_results) / len(ablation_results) * 100
abl_f1   = sum(r["f1"]       for r in ablation_results) / len(ablation_results)
abl_abs  = sum(r["abstained"] for r in ablation_results) / len(ablation_results) * 100

print("\n" + "═" * 60)
print("ABLATION RESULTS: Does query refinement help?")
print("═" * 60)
print(f"{'Metric':<25} {'Full Adaptive':>15} {'No Refinement':>15} {'Delta':>10}")
print("─" * 60)
print(f"{'Exact Match (%)':25} {full_em:>15.1f} {abl_em:>15.1f} {full_em-abl_em:>+10.1f}")
print(f"{'F1 Score':25} {full_f1:>15.3f} {abl_f1:>15.3f} {full_f1-abl_f1:>+10.3f}")
print(f"{'Abstention Rate (%)':25} {full_abs:>15.1f} {abl_abs:>15.1f} {full_abs-abl_abs:>+10.1f}")
print("═" * 60)

interpretation = "✅ Query refinement HELPS" if (full_em - abl_em) > 2.0 else \
                 "⚠️  Query refinement has MARGINAL effect" if (full_em - abl_em) > 0 else \
                 "❌ Query refinement HURTS or has no effect"
print(f"\nConclusion: {interpretation}")
print("(A positive delta means full adaptive outperforms no-refinement)")
